# Dia5 - Bronze + Gold: Indian E-Commerce (Dimensões e Bronze)

**Objetivo:** Criar as tabelas Bronze (`customers`, `products`, `sales`) a partir dos CSVs do dataset Indian E-Commerce e construir as dimensões Gold (`dim_datas`, `dim_horas`, `dim_produtos`, `dim_clientes`) com chaves substitutas (SK) e MERGE para idempotência.

| Item | Valor |
|---|---|
| **Camada** | Bronze + Gold (dimensões) |
| **Entrada** | `/Volumes/{catalog}/landing/files/indian/*.csv` |
| **Saída** | `{catalog}.bronze.{customers,products,sales}`, `{catalog}.gold.{dim_datas,dim_horas,dim_produtos,dim_clientes}`, `{catalog}.silver.{dim_produtos,dim_clientes}` |
| **Ordem no pipeline** | Após `Dia6a-Gerador-CSVs-Indian` (geração dos CSVs) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Bronze — Carga dos CSVs

Cria as 3 tabelas Bronze a partir dos CSVs do dataset Indian E-Commerce. `CREATE OR REPLACE TABLE` com `read_files` garante idempotência.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`customers`
    SELECT * EXCEPT(_rescued_data), from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
    FROM read_files('/Volumes/${catalog}/landing/files/indian/customers.csv',
        format => 'csv',
        inferColumnTypes => True,
        header => True
    );

CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`products`
    SELECT * EXCEPT(_rescued_data), from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
    FROM read_files('/Volumes/${catalog}/landing/files/indian/products.csv',
        format => 'csv',
        inferColumnTypes => True,
        header => True
    );

CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`sales`
    SELECT * EXCEPT(_rescued_data), from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
    FROM read_files('/Volumes/${catalog}/landing/files/indian/sales.csv',
        format => 'csv',
        inferColumnTypes => True,
        header => True
    );

## Gold — Dimensão de Datas (`dim_datas`)

Cria a dimensão de datas com intervalo 2024-01-01 a 2026-12-31. `CREATE OR REPLACE TABLE` (DDL) + `INSERT` para popular a dimensão.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`gold`.`dim_datas` ( 
	dat_sk_data BIGINT NOT NULL COMMENT 'Primary key e Surrogate key da tabela',
	dat_data DATE NOT NULL COMMENT 'Data no formato YYYY-MM-DD',
	dat_nu_dia SMALLINT NOT NULL COMMENT 'Número do dia no mês',
	dat_nu_mes SMALLINT NOT NULL COMMENT 'Número do mês',
	dat_nu_ano SMALLINT NOT NULL COMMENT 'Ano',
	dat_nu_trimestre SMALLINT NOT NULL COMMENT 'Número do trimestre',
	dat_nm_dia_semana STRING NOT NULL COMMENT 'Nome do dia na semana',
	dat_dh_atualizacao TIMESTAMP DEFAULT from_utc_timestamp(now(), 'GMT-3') COMMENT 'Data/hora da última atualização da linha',
	    CONSTRAINT pk_dim_datas PRIMARY KEY ( dat_sk_data )
 ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported',
                    'delta.logRetentionDuration'='7 days',
					'delta.autoOptimize.autoCompact'='auto',
                    'spark.databricks.delta.autoCompact.enabled'='true');

INSERT INTO `${catalog}`.`gold`.`dim_datas` (dat_sk_data, dat_data, dat_nu_dia, dat_nu_mes, dat_nu_ano, dat_nu_trimestre, dat_nm_dia_semana)
    WITH date_range AS (
        SELECT sequence(DATE('2024-01-01'), DATE('2026-12-31'), INTERVAL 1 DAY) AS dates
    ),
    expanded AS (
        SELECT EXPLODE(dates) AS dat_data FROM date_range
    )
SELECT
    CAST(DATE_FORMAT(dat_data, 'yyyyMMdd') AS BIGINT) AS dat_sk_data,
    dat_data,
    CAST(DAY(dat_data) AS SMALLINT) AS dat_nu_dia,
    CAST(MONTH(dat_data) AS SMALLINT) AS dat_nu_mes,
    CAST(YEAR(dat_data) AS SMALLINT) AS dat_nu_ano,
    CAST(QUARTER(dat_data) AS SMALLINT) AS dat_nu_trimestre,
    CASE DAYOFWEEK(dat_data)
        WHEN 1 THEN 'Domingo' WHEN 2 THEN 'Segunda-feira' WHEN 3 THEN 'Terça-feira'
        WHEN 4 THEN 'Quarta-feira' WHEN 5 THEN 'Quinta-feira' WHEN 6 THEN 'Sexta-feira' WHEN 7 THEN 'Sábado'
    END AS dat_nm_dia_semana
FROM expanded ORDER BY dat_data;

## Gold — Dimensão de Horas (`dim_horas`)

Cria a DDL da tabela e insere a linha -1 ("Não Informado"). Depois, um script Python gera as 1440 linhas (24h × 60min) e as carrega via overwrite.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`gold`.`dim_horas` ( 
	hor_sk_hora SMALLINT NOT NULL COMMENT 'Primary key e Surrogate key da tabela',
	hor_id_hora STRING NOT NULL COMMENT 'Identificador exclusivo para cada hora+minuto do dia',
	hor_nu_hora SMALLINT NOT NULL COMMENT 'Valor númerico para hora (0-23)',
	hor_nu_minuto SMALLINT NOT NULL COMMENT 'Valor númerico para minuto (0-59)',
	hor_ds_periodo_dia STRING NOT NULL COMMENT 'Descrição do período do dia',
	hor_dt_atualizacao TIMESTAMP NOT NULL DEFAULT from_utc_timestamp(now(), 'GMT-3') COMMENT 'Data/hora da última atualização da linha',
	    CONSTRAINT pk_dim_horas PRIMARY KEY ( hor_sk_hora )
) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported',
                    'delta.logRetentionDuration'='7 days',
					'delta.autoOptimize.autoCompact'='auto',
                    'spark.databricks.delta.autoCompact'='true');

In [0]:
import pandas as pd
import numpy as np

periodo = '0'
countHora = 0
i = 1
lst0, lst1, lst2, lst3, lst4 = [], [], [], [], []

while countHora <= 23:
    hora = countHora
    countMinuto = 0
    numHora = 2 - len(str(hora))
    while countMinuto < 60:
        minuto = countMinuto
        numMinuto = 2 - len(str(minuto))
        id_hora = (('0' * numHora) + str(hora)) + (('0' * numMinuto) + str(minuto))
        if 0 <= hora < 6: periodo = 'Madrugada'
        elif 6 <= hora < 12: periodo = 'Manhã'
        elif 12 <= hora < 18: periodo = 'Tarde'
        elif 18 <= hora < 24: periodo = 'Noite'
        lst0.insert(i, i); lst1.insert(i, id_hora); lst2.insert(i, hora); lst3.insert(i, minuto); lst4.insert(i, periodo)
        countMinuto += 1; i += 1
    countHora += 1

df_DimHora = pd.DataFrame(list(zip(lst0, lst1, lst2, lst3, lst4)), columns=['hor_sk_hora','hor_id_hora','hor_nu_hora','hor_nu_minuto','hor_ds_periodo_dia'])
d = dict.fromkeys(df_DimHora.select_dtypes(np.int64).columns, np.int16)
df_DimHora = df_DimHora.astype(d)

df_DimHoraSpark = spark.createDataFrame(df_DimHora)
df_DimHoraSpark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.gold.dim_horas")

## Gold — Dimensão de Produtos (`dim_produtos`)

Cria a DDL com linha -1 ("Não Informado"), cria a Silver intermediária com `CREATE OR REPLACE TABLE` a partir de `bronze.products`, e faz `MERGE INTO` na Gold para idempotência (insere novos, atualiza existentes).

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`gold`.`dim_produtos` ( 
	pro_sk_produto BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY ( START WITH 1 ) COMMENT 'Primary key e Surrogate key da tabela',
	pro_id_produto STRING NOT NULL COMMENT 'ID do produto no sistema origem',
	pro_nm_produto STRING NOT NULL COMMENT 'Nome do produto',
	pro_ds_categoria STRING NOT NULL COMMENT 'Categoria do produto',
	pro_nm_marca STRING NOT NULL COMMENT 'Nome da marca',
	pro_dh_atualizacao TIMESTAMP NOT NULL DEFAULT from_utc_timestamp(now(), 'GMT-3') COMMENT 'Data/hora da última atualização da linha',
	    CONSTRAINT pk_dim_produtos PRIMARY KEY ( pro_sk_produto )
 ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported',
                    'delta.logRetentionDuration'='7 days',
					'delta.autoOptimize.autoCompact'='auto',
                    'spark.databricks.delta.autoCompact'='true');

CREATE OR REPLACE TABLE `${catalog}`.`silver`.`dim_produtos` AS
    SELECT DISTINCT
        TRIM(Product_ID) AS pro_id_produto,
        TRIM(Product_Name) AS pro_nm_produto,
        TRIM(Category) AS pro_ds_categoria,
        TRIM(Brand) AS pro_nm_marca,
        from_utc_timestamp(now(), 'GMT-3') AS pro_dh_atualizacao
    FROM `${catalog}`.`bronze`.`products`;

MERGE INTO `${catalog}`.`gold`.`dim_produtos` AS target
    USING `${catalog}`.`silver`.`dim_produtos` AS source
        ON (target.pro_id_produto = source.pro_id_produto)
    WHEN MATCHED THEN
        UPDATE SET
            pro_nm_produto      = source.pro_nm_produto,
            pro_ds_categoria    = source.pro_ds_categoria,
            pro_nm_marca        = source.pro_nm_marca,
            pro_dh_atualizacao  = from_utc_timestamp(now(), 'GMT-3')
    WHEN NOT MATCHED THEN
        INSERT (pro_id_produto, pro_nm_produto, pro_ds_categoria, pro_nm_marca)
        VALUES (source.pro_id_produto, source.pro_nm_produto, source.pro_ds_categoria, source.pro_nm_marca);

## Gold — Dimensão de Clientes (`dim_clientes`)

Mesmo padrão: DDL com linha -1, Silver intermediária a partir de `bronze.customers`, e `MERGE INTO` na Gold para idempotência.

> **⚠️ Atenção:** A coluna `cli_ds_email` contém e-mails de exemplo (ex: `aarav.sharma@example.com`) — são dados fictícios do dataset Indian E-Commerce, não dados reais.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`gold`.`dim_clientes` ( 
	cli_sk_cliente BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY ( START WITH 1  ) COMMENT 'Primary key e Surrogate key da tabela',
	cli_id_cliente STRING NOT NULL COMMENT 'ID do cliente',
	cli_nm_nome STRING NOT NULL COMMENT 'Nome do cliente',
	cli_ds_sexo STRING NOT NULL COMMENT 'Sexo do cliente',
	cli_dt_nascimento DATE NOT NULL COMMENT 'Data de nascimento do cliente',
	cli_ds_email STRING NOT NULL COMMENT 'Email do cliente',
	cli_nu_telefone STRING NOT NULL COMMENT 'Número de telefone do cliente',
	cli_dt_cadastro DATE NOT NULL COMMENT 'Data de cadastro do cliente',
	cli_ds_categoria STRING NOT NULL COMMENT 'Categoria do cliente',
	cli_dh_atualizacao TIMESTAMP NOT NULL DEFAULT from_utc_timestamp(now(), 'GMT-3') COMMENT 'Data/hora da última atualização da linha',
	    CONSTRAINT pk_dim_clientes PRIMARY KEY ( cli_sk_cliente )
 ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported',
                    'delta.logRetentionDuration'='7 days',
					'delta.autoOptimize.autoCompact'='auto',
                    'spark.databricks.delta.autoCompact'='true');

CREATE OR REPLACE TABLE `${catalog}`.`silver`.`dim_clientes` AS
    SELECT DISTINCT
        TRIM(Customer_ID) AS cli_id_cliente,
        TRIM(Customer_Name) AS cli_nm_nome,
        CASE WHEN TRIM(Gender) = 'male' THEN 'Masculino' ELSE 'Feminino' END AS cli_ds_sexo,
        TRIM(Date_of_Birth) AS cli_dt_nascimento,
        Email AS cli_ds_email,
        Phone AS cli_nu_telefone,
        Registration_Date AS cli_dt_cadastro,
        Customer_Tier AS cli_ds_categoria,
        from_utc_timestamp(now(), 'GMT-3') AS cli_dh_atualizacao
    FROM `${catalog}`.`bronze`.`customers`;

MERGE INTO `${catalog}`.`gold`.`dim_clientes` AS target
    USING `${catalog}`.`silver`.`dim_clientes` AS source
        ON (target.cli_id_cliente = source.cli_id_cliente)
    WHEN MATCHED THEN
        UPDATE SET
            cli_nm_nome       = source.cli_nm_nome,
            cli_ds_sexo       = source.cli_ds_sexo,
            cli_dt_nascimento = source.cli_dt_nascimento,
            cli_ds_email      = source.cli_ds_email,
            cli_nu_telefone   = source.cli_nu_telefone,
            cli_dt_cadastro   = source.cli_dt_cadastro,
            cli_dh_atualizacao  = from_utc_timestamp(now(), 'GMT-3')
    WHEN NOT MATCHED THEN
        INSERT (cli_id_cliente, cli_nm_nome, cli_ds_sexo, cli_dt_nascimento, cli_ds_email, cli_nu_telefone, cli_dt_cadastro, cli_ds_categoria)
        VALUES (source.cli_id_cliente, source.cli_nm_nome, source.cli_ds_sexo, source.cli_dt_nascimento, source.cli_ds_email, source.cli_nu_telefone, source.cli_dt_cadastro, source.cli_ds_categoria);